# KV-Edit → Modular Diffusers — smoke (training-free editing, precise background preservation)

Masked text editing on FLUX with cached background K/V (KV-Edit, arXiv:2502.17363). Publish PRIVATE `remyxai/kv-edit-flux-modular` → load via `trust_remote_code` → assert `KVEditBlock` → a tiny masked edit → the empty-mask no-op spike. Upload `block.py` first. Runtime: A100 · `HUGGINGFACE_TOKEN` · accept FLUX.1-dev.


## 1 · Install + GPU + auth


In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf scikit-image


In [ ]:
import torch
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16


## 2 · Publish PRIVATE (upload block.py first)


In [ ]:
import os, json
from huggingface_hub import HfApi
assert os.path.exists("block.py"), "Upload block.py first."
open("modular_config.json","w").write(json.dumps({"_class_name":"KVEditBlock","_diffusers_version":"0.41.0.dev0","auto_map":{"ModularPipelineBlocks":"block.KVEditBlock"}},indent=2))
F="black-forest-labs/FLUX.1-dev"
def c(s,l,cl): return [None,None,{"pretrained_model_name_or_path":F,"revision":None,"subfolder":s,"type_hint":[l,cl],"variant":None}]
open("modular_model_index.json","w").write(json.dumps({"_blocks_class_name":"KVEditBlock","_class_name":"ModularPipeline","_diffusers_version":"0.41.0.dev0",
 "text_encoder":c("text_encoder","transformers","CLIPTextModel"),"tokenizer":c("tokenizer","transformers","CLIPTokenizer"),
 "text_encoder_2":c("text_encoder_2","transformers","T5EncoderModel"),"tokenizer_2":c("tokenizer_2","transformers","T5TokenizerFast"),
 "transformer":c("transformer","diffusers","FluxTransformer2DModel"),"vae":c("vae","diffusers","AutoencoderKL"),
 "scheduler":c("scheduler","diffusers","FlowMatchEulerDiscreteScheduler")},indent=2))
api=HfApi(); REPO="remyxai/kv-edit-flux-modular"; api.create_repo(REPO,private=True,repo_type="model",exist_ok=True)
for f in ["block.py","modular_config.json","modular_model_index.json"]: api.upload_file(path_or_fileobj=f,path_in_repo=f,repo_id=REPO)
print("published PRIVATE:", api.list_repo_files(REPO))


## 3 · Load + a source image + mask


In [ ]:
from diffusers import ModularPipeline
from PIL import Image, ImageDraw
from io import BytesIO
import requests
from IPython.display import display
pipe = ModularPipeline.from_pretrained("remyxai/kv-edit-flux-modular", trust_remote_code=True)
assert type(pipe.blocks).__name__ == "KVEditBlock", type(pipe.blocks).__name__
print("loaded block:", type(pipe.blocks).__name__)   # expect KVEditBlock
pipe.load_components(dtype=DT); pipe.to(DEV)

IMG_URL = "https://raw.githubusercontent.com/fallenshock/FlowEdit/main/inputs/cat.png"  #@param {type:"string"}
try:
    src = Image.open(BytesIO(requests.get(IMG_URL, timeout=30).content)).convert("RGB")
except Exception as e:
    print("fetch failed, upload one:", e)
    from google.colab import files; up=files.upload(); src=Image.open(list(up.keys())[0]).convert("RGB")
src = src.resize((1024,1024)); src.save("src.png")

# default edit mask: a box over the central subject (white = edit, black = keep). Upload your own for real use.
mask = Image.new("L", (1024,1024), 0)
ImageDraw.Draw(mask).rectangle([256, 256, 768, 768], fill=255)
mask.save("mask.png")
print("source | mask:"); display(src.resize((320,320))); display(mask.resize((320,320)))


## 4 · Milestone A — smoke (tiny masked edit)


In [ ]:
import torch
g = torch.Generator(DEV).manual_seed(0)
sm = pipe(image="src.png", mask="mask.png", source_prompt="a cat", prompt="a dog",
          height=512, width=512, T_steps=12, generator=g).images[0]
assert sm.size == (512, 512), sm.size
print("[SMOKE] ran; output size", sm.size)
from IPython.display import display; display(sm)


## 5 · Milestone B — no-op spike (the K/V seam)

All-keep mask → the denoise substitutes cached K/V for **every** image token → output reconstructs the source (low L1). `mask=None` → seam disarmed (bit-exact stock FLUX attention) → still runs. This gates the capture/substitute seam before the full e2e.


In [ ]:
import numpy as np, torch
from PIL import Image
# empty mask (all keep) -> the denoise runs on cached K/V for EVERY token -> reconstructs the source.
keep = Image.new("L", (1024,1024), 0); keep.save("keep_all.png")
g = torch.Generator(DEV).manual_seed(0)
recon = pipe(image="src.png", mask="keep_all.png", source_prompt="a cat", prompt="a dog",
             height=512, width=512, T_steps=12, generator=g).images[0]
a = np.asarray(recon.resize((512,512))).astype(np.float32)/255.0
b = np.asarray(src.resize((512,512))).astype(np.float32)/255.0
l1 = float(np.abs(a-b).mean())
print(f"[NO-OP SPIKE] empty-edit-mask reconstruction L1 = {l1:.4f} (low = background tokens preserved)")
# mask=None -> whole-image edit, seam disarmed (stock attention path) -> should just run.
g = torch.Generator(DEV).manual_seed(0)
whole = pipe(image="src.png", mask=None, source_prompt="a cat", prompt="a dog",
             height=512, width=512, T_steps=12, generator=g).images[0]
print("[NO-OP SPIKE] mask=None (disarmed) ran; output size", whole.size)


## Verdict
`loaded block: KVEditBlock` + a coherent tiny edit + a low empty-mask reconstruction L1 = the modular KV-Edit seam works. Then run `e2e.ipynb` for the quantitative background-preservation check + FlowEdit comparison.
